In [0]:
from pyspark.sql.functions import col, monotonically_increasing_id, count, avg, round as _round, monotonically_increasing_id

spark.sql("CREATE SCHEMA IF NOT EXISTS rocket_lab.gold")

In [0]:
df = spark.table("rocket_lab.silver.tb_info_filmes")

df_dim = df.withColumn("sk_movie_id", monotonically_increasing_id().cast("bigint")) \
    .select(
        col("sk_movie_id"),
        col("id_filme"),
        col("titulo"),
        col("data_lancamento"),
        col("ano_lancamento"),
        col("duracao_minutos"),
        col("idioma_original"),
        col("status_filme"),
        col("sinopse")
    )

df_dim.write.mode("overwrite").saveAsTable("rocket_lab.gold.dim_movies")

In [0]:
df = spark.table("rocket_lab.silver.tb_generos")

df_dim = (
    df.select("nome_genero").distinct()
      .withColumn("sk_genre_id", monotonically_increasing_id().cast("bigint"))
      .select("sk_genre_id", "nome_genero")
)

df_dim.write.mode("overwrite").saveAsTable("rocket_lab.gold.dim_genres")

In [0]:
df = spark.table("rocket_lab.silver.tb_pessoas_empresas")

df_dim = (
    df.select("nome_entidade", "tipo_entidade")
      .filter(col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
      .distinct()
      .withColumn("sk_person_id", monotonically_increasing_id().cast("bigint"))
      .select(
          col("sk_person_id"),
          col("nome_entidade").alias("nome_pessoa"),
          col("tipo_entidade").alias("tipo_pessoa")
      )
)

df_dim.write.mode("overwrite").saveAsTable("rocket_lab.gold.dim_people")

In [0]:
df = spark.table("rocket_lab.silver.tb_pessoas_empresas")

df_dim = (
    df.filter(col("tipo_entidade") == "Produtora")
      .select(col("nome_entidade").alias("nome_produtora"))
      .distinct()
      .withColumn("sk_company_id", monotonically_increasing_id().cast("bigint"))
      .select("sk_company_id", "nome_produtora")
)

df_dim.write.mode("overwrite").saveAsTable("rocket_lab.gold.dim_companies")

In [0]:
reviews = spark.table("rocket_lab.silver.tb_avaliacoes_usuarios")
dim_movies = spark.table("rocket_lab.gold.dim_movies").select("sk_movie_id", "id_filme")

df = (
    reviews.groupBy("id_filme")
        .agg(
            count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
            _round(avg("nota_usuario"), 2).alias("nota_media_usuarios")
        )
)

# join pra pegar o SK
df = df.join(dim_movies, "id_filme")

df = df.withColumn("sk_review_id", monotonically_increasing_id().cast("bigint")) \
    .select(
        col("sk_review_id"),
        col("sk_movie_id"),
        col("qtd_avaliacoes_usuarios"),
        col("nota_media_usuarios")
    )

df.write.mode("overwrite").saveAsTable("rocket_lab.gold.dim_reviews")

In [0]:
generos = spark.table("rocket_lab.silver.tb_generos")
dim_movies = spark.table("rocket_lab.gold.dim_movies").select("sk_movie_id", "id_filme")
dim_genres = spark.table("rocket_lab.gold.dim_genres")

df = (
    generos
    .join(dim_movies, "id_filme")
    .join(dim_genres, "nome_genero")
    .select("sk_movie_id", "sk_genre_id")
    .distinct()
)

df.write.mode("overwrite").saveAsTable("rocket_lab.gold.bridge_movie_genre")

In [0]:
pessoas = spark.table("rocket_lab.silver.tb_pessoas_empresas")
dim_movies = spark.table("rocket_lab.gold.dim_movies").select("sk_movie_id", "id_filme")
dim_people = spark.table("rocket_lab.gold.dim_people")

df = (
    pessoas
    .filter(col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .join(dim_movies, "id_filme")
    .join(
        dim_people,
        (pessoas.nome_entidade == dim_people.nome_pessoa) &
        (pessoas.tipo_entidade == dim_people.tipo_pessoa)
    )
    .select("sk_movie_id", "sk_person_id")
    .distinct()
)

df.write.mode("overwrite").saveAsTable("rocket_lab.gold.bridge_movie_person")

In [0]:
produtoras = spark.table("rocket_lab.silver.tb_pessoas_empresas").filter(col("tipo_entidade") == "Produtora")
dim_movies = spark.table("rocket_lab.gold.dim_movies").select("sk_movie_id", "id_filme")
dim_companies = spark.table("rocket_lab.gold.dim_companies")

df = (
    produtoras
    .join(dim_movies, "id_filme")
    .join(dim_companies, produtoras.nome_entidade == dim_companies.nome_produtora)
    .select("sk_movie_id", "sk_company_id")
    .distinct()
)

df.write.mode("overwrite").saveAsTable("rocket_lab.gold.bridge_movie_company")

In [0]:
dim_movies = spark.table("rocket_lab.gold.dim_movies").select("sk_movie_id", "id_filme")
fin = spark.table("rocket_lab.silver.tb_financeiro_filmes")
eng = spark.table("rocket_lab.silver.tb_metricas_engajamento")

df = (
    dim_movies
    .join(fin, "id_filme", "left")
    .join(eng, "id_filme", "left")
    .select(
        col("sk_movie_id"),
        col("orcamento_usd").cast("decimal(18,2)"),
        col("receita_usd").cast("decimal(18,2)"),
        col("lucro_usd").cast("decimal(18,2)"),
        col("orcamento_brl").cast("decimal(18,2)"),
        col("receita_brl").cast("decimal(18,2)"),
        col("lucro_brl").cast("decimal(18,2)"),
        col("popularidade").cast("double"),
        col("nota_media_tmdb").cast("double"),
        col("qtd_votos_tmdb").cast("int"),
        col("nota_media_imbd").cast("double"),
        col("qtd_votos_imbd").cast("int")
    )
)

df.write.mode("overwrite").saveAsTable("rocket_lab.gold.fact_movies_performance")

In [0]:
%sql
SELECT SUM(receita_brl) AS receita_total_brl
FROM rocket_lab.gold.fact_movies_performance;

In [0]:
%sql
SELECT m.titulo, f.popularidade
FROM rocket_lab.gold.fact_movies_performance f
JOIN rocket_lab.gold.dim_movies m ON m.sk_movie_id = f.sk_movie_id
WHERE f.popularidade IS NOT NULL
ORDER BY f.popularidade DESC
LIMIT 5;

In [0]:
%sql
SELECT g.nome_genero, COUNT(DISTINCT b.sk_movie_id) AS qtd_filmes
FROM rocket_lab.gold.bridge_movie_genre b
JOIN rocket_lab.gold.dim_genres g ON g.sk_genre_id = b.sk_genre_id
GROUP BY g.nome_genero
ORDER BY qtd_filmes DESC;

In [0]:
%sql
SELECT
    m.titulo,
    f.receita_usd,
    f.receita_brl,
    RANK() OVER (ORDER BY f.receita_usd DESC) AS posicao
FROM rocket_lab.gold.fact_movies_performance f
JOIN rocket_lab.gold.dim_movies m ON m.sk_movie_id = f.sk_movie_id
WHERE f.receita_usd IS NOT NULL
ORDER BY posicao
LIMIT 10;

In [0]:
%sql
WITH limite AS (
    SELECT MAX(data_lancamento) AS data_max
    FROM rocket_lab.gold.dim_movies
    WHERE data_lancamento <= CURRENT_DATE()
),
recorte AS (
    SELECT m.sk_movie_id
    FROM rocket_lab.gold.dim_movies m, limite l
    WHERE m.data_lancamento BETWEEN ADD_MONTHS(l.data_max, -24) AND l.data_max
)
SELECT p.nome_pessoa, COUNT(*) AS participacoes
FROM rocket_lab.gold.bridge_movie_person bp
JOIN rocket_lab.gold.dim_people p ON p.sk_person_id = bp.sk_person_id
JOIN recorte r ON r.sk_movie_id = bp.sk_movie_id
WHERE p.tipo_pessoa = 'Ator'
GROUP BY p.nome_pessoa
ORDER BY participacoes DESC
LIMIT 1;

In [0]:
%sql
WITH limite AS (
    SELECT MAX(data_lancamento) AS data_max
    FROM rocket_lab.gold.dim_movies
    WHERE data_lancamento <= CURRENT_DATE()
),
recorte AS (
    SELECT m.sk_movie_id
    FROM rocket_lab.gold.dim_movies m, limite l
    WHERE m.data_lancamento BETWEEN ADD_MONTHS(l.data_max, -60) AND l.data_max
)
SELECT c.nome_produtora, SUM(f.lucro_usd) AS lucro_total_usd
FROM rocket_lab.gold.bridge_movie_company bc
JOIN rocket_lab.gold.dim_companies c ON c.sk_company_id = bc.sk_company_id
JOIN rocket_lab.gold.fact_movies_performance f ON f.sk_movie_id = bc.sk_movie_id
JOIN recorte r ON r.sk_movie_id = bc.sk_movie_id
WHERE f.lucro_usd IS NOT NULL
GROUP BY c.nome_produtora
ORDER BY lucro_total_usd DESC
LIMIT 1;